In [ ]:
import numpy as np
import torch
from occhio import ToyModel, ModelGrid
from occhio.autoencoder import TiedLinearRelu, SynthAE
from occhio.distributions import SparseUniform
from occhio.model_grid import Axis
from sae_lens import (
    StandardTrainingSAE,
    StandardTrainingSAEConfig,
    MatryoshkaBatchTopKTrainingSAE,
    MatryoshkaBatchTopKTrainingSAEConfig,
    BatchTopKTrainingSAE,
    BatchTopKTrainingSAEConfig,
    JumpReLUTrainingSAE,
    JumpReLUTrainingSAEConfig,
    MatchingPursuitTrainingSAE,
    MatchingPursuitTrainingSAEConfig,
)
from occhio.visualization_2 import RepresentationPlot
from occhio.visualization_2.core import CompositePlot
from occhio.visualization_2.plots import SAEClassificationMetricsPlot
from occhio.visualization_2.plots.sae_classification_metric import (
    SAEClassificationMetric,
    SAEClassificationMetricPlot,
    SAEMetricsComparisonPlot,
)

In [ ]:
DEVICE = "mps"

In [ ]:
# Monkey-patch for MPS compatibility: BatchTopKTrainingSAE uses float64 for topk_threshold,
# which MPS doesn't support. We patch torch.tensor to downgrade float64 to float32 on MPS.
import torch

_original_torch_tensor = torch.tensor


def _patched_torch_tensor(*args, **kwargs):
    device = kwargs.get("device", None)
    dtype = kwargs.get("dtype", None)
    # If creating float64 tensor on MPS, use float32 instead
    if dtype == torch.float64 or dtype == torch.double:
        device_str = str(device) if device is not None else ""
        if "mps" in device_str:
            kwargs["dtype"] = torch.float32
    return _original_torch_tensor(*args, **kwargs)


torch.tensor = _patched_torch_tensor

In [ ]:
N_FEATURES = 200
N_HIDDEN = 40

In [ ]:
P_MAX = 0.4
P_MIN = 0.5 / N_FEATURES
ALPHA = 0.5

ranks = np.arange(1, N_FEATURES + 1, dtype=np.float64)
raw = ranks ** (-ALPHA)
# Rescale so that rank 1 → P_MAX, rank N → P_MIN
firing_probs = P_MIN + (P_MAX - P_MIN) * (raw - raw[-1]) / (raw[0] - raw[-1])
firing_probs = torch.tensor(firing_probs, dtype=torch.float32)

In [ ]:
from enum import Enum


# [2026-03-19 | OliverSieweke] TODO: Define this in occhio directly.
class AutoencoderType(Enum):
    TiedLinearRelu = TiedLinearRelu.__name__
    SynthAE = SynthAE.__name__

In [ ]:
def create_model(params):
    generator = torch.Generator(device=DEVICE).manual_seed(199)

    match params["Autoencoder"]:
        case AutoencoderType.TiedLinearRelu:
            ae = TiedLinearRelu(
                N_FEATURES, N_HIDDEN, device=DEVICE, generator=generator
            )
        case AutoencoderType.SynthAE:
            ae = SynthAE(N_FEATURES, N_HIDDEN, device=DEVICE, generator=generator)

    return ToyModel(
        ae=ae,
        distribution=SparseUniform(
            N_FEATURES, p_active=firing_probs, device=DEVICE, generator=generator
        ),
        device=DEVICE,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(
            label="Autoencoder",
            values=[AutoencoderType.SynthAE, AutoencoderType.TiedLinearRelu],
        ),
    ],
)

In [ ]:
grid.fit(15000)

In [ ]:
standard_sae_config = StandardTrainingSAEConfig(
    d_in=N_HIDDEN,
    d_sae=N_FEATURES,
    l1_coefficient=5e-2,
    device=DEVICE,
)

matryoshka_sae_config = MatryoshkaBatchTopKTrainingSAEConfig(
    d_in=N_HIDDEN,
    d_sae=N_FEATURES,
    k=50,  # number of active features
    matryoshka_widths=[50, 100, 200],  # final must equal d_sae=200
    device=DEVICE,
)

batch_topk_sae_config = BatchTopKTrainingSAEConfig(
    d_in=N_HIDDEN,
    d_sae=N_FEATURES,
    k=50,  # number of active features per batch element
    device=DEVICE,
)

jumprelu_sae_config = JumpReLUTrainingSAEConfig(
    d_in=N_HIDDEN,
    d_sae=N_FEATURES,
    device=DEVICE,
)

matching_pursuit_sae_config = MatchingPursuitTrainingSAEConfig(
    d_in=N_HIDDEN,
    d_sae=N_FEATURES,
    max_iterations=50,  # number of pursuit iterations
    device=DEVICE,
)

standard_sae = StandardTrainingSAE(standard_sae_config)
matryoshka_sae = MatryoshkaBatchTopKTrainingSAE(matryoshka_sae_config)
batch_topk_sae = BatchTopKTrainingSAE(batch_topk_sae_config)
jumprelu_sae = JumpReLUTrainingSAE(jumprelu_sae_config)
matching_pursuit_sae = MatchingPursuitTrainingSAE(matching_pursuit_sae_config)

grid.train_saes(
    {
        "Standard": standard_sae,
        "Matryoshka": matryoshka_sae,
        "BatchTopK": batch_topk_sae,
        "JumpReLU": jumprelu_sae,
        "MatchingPursuit": matching_pursuit_sae,
    },
    # training_samples=1000,
    verbose=True,
)

In [ ]:
grid.evaluate_saes(verbose=True)

In [ ]:
plot = SAEClassificationMetricsPlot(group_by="sae")
plot(grid, height=800)

In [ ]:
plot_2 = SAEClassificationMetricPlot(SAEClassificationMetric.ACCURACY)
plot_2(grid, height=800)

In [ ]:
composite_plot = CompositePlot(
    layout=[
        [
            SAEClassificationMetricPlot(SAEClassificationMetric.PRECISION),
            SAEClassificationMetricPlot(SAEClassificationMetric.RECALL),
        ],
        [
            SAEClassificationMetricPlot(SAEClassificationMetric.ACCURACY),
            SAEClassificationMetricPlot(SAEClassificationMetric.F1),
        ],
    ],
)

composite_plot(grid, height=800)

# [2026-03-19 | OliverSieweke] TODO: add x axis description
# [2026-03-19 | OliverSieweke] TODO: add title with number of features etc.

In [ ]:
composite_plot_2 = CompositePlot(
    layout=[
        [
            SAEMetricsComparisonPlot(sae_label="Standard"),
            SAEMetricsComparisonPlot(sae_label="MatchingPursuit"),
        ],
        [
            SAEMetricsComparisonPlot(sae_label="BatchTopK"),
            SAEMetricsComparisonPlot(sae_label="JumpReLU"),
        ],
        [SAEMetricsComparisonPlot(sae_label="Matryoshka"), None],
    ],
)
composite_plot_2(grid, height=800)

In [ ]:
import importlib
import occhio.visualization_2.core.figure_wrappers
import occhio.visualization_2.core.base_plot
import occhio.visualization_2.plots.representation

importlib.reload(occhio.visualization_2.core.figure_wrappers)
importlib.reload(occhio.visualization_2.core.base_plot)
importlib.reload(occhio.visualization_2.plots.representation)

from occhio.visualization_2.plots.representation import RepresentationPlot

plot = RepresentationPlot()

plot(grid)

In [ ]:
# Check superposition
